# The parser function

The parser will take the raw JSON files and produce dictionaries, rows, clean data, .csv files of information that we need. Then, the data will be saved in the data/processed folder

In [28]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [29]:
import json
import requests

from config import STATIONS
from datetime import datetime, timedelta, timezone
from pprint import pprint

import csv
import pandas as pd


Let's read in one of the .json files. EDIT LATER TO READ ALL THE JSON FILES:

In [30]:
with open('/home/armankorajac/flixbus_tracker/data/raw/genoa_principe_departures_2026_05_13T09_13_to_2026_05_13T17_13.json', 'r') as f:
    data = json.load(f)
    pprint(data)

{'rides': [{'calls': [{'arrival': None,
                       'departure': {'deviation': None,
                                     'platform': None,
                                     'scheduled': '2026-05-12T18:45:00Z'},
                       'sequence': 1,
                       'stop': {'city': {'id': '40df577e-8646-11e6-9066-549f350fcb0c',
                                         'name': 'Bordeaux'},
                                'code': 'BDX',
                                'description': 'Paludate bus station is '
                                               'located at Quai de Paludate, '
                                               'between the SNCF railway '
                                               'bridge and the MetPark '
                                               'Paludate parking.',
                                'id': 'dcc06cc2-9603-11e6-9066-549f350fcb0c',
                                'location': {'latitude': 44.82975,
                          

The most important arrays are:
1) data['rides'][n]['id']  
2) data['rides'][n]['status']. \

From there, data['rides'][n]['status']['deviation']['deviation_seconds'] is important for the deviation. \
General information are available at data['rides'][n]['line'] (for example 'direction' or 'code' or 'name' given the initial stop - final stop info). Here, n is the number of _rides_ that are in the .json file. In principle it's okay when data overlap. We need anyway the information of the deviations for different timestamps and stuff. 

In [31]:
data['rides'][2]['line']

{'code': '798',
 'direction': 'Marseille (Saint-Charles)',
 'name': 'Marseille - Milan',
 'trip_number': 1,
 'means_of_transport': 'BUS',
 'brand': {'id': 'a18f138c-68fa-4b45-a42f-adb0378e10d3', 'name': 'FlixBus'}}

In [32]:
print(len(data['rides']))

18


In [33]:
data['rides'][0]['status']

{'segment': None,
 'next_stop_sequence': None,
 'has_arrived_at_next_stop': True,
 'progress': None,
 'deviation': {'deviation_timestamp': '2026-05-13T09:55:17Z',
  'deviation_seconds': 3617,
  'reason': None,
  'deviation_class': 'LATE',
  'deviation_type': 'ACTUAL',
  'updated_at': '2026-05-13T09:58:33Z'},
 'scheduled_timestamp': '2026-05-13T08:55:00Z'}

In [ ]:
def extract_dep_delay_id(data, n): # Extract data for the ID of the bus, general information, delay (if given), when it was updated, and the scheduled departure time

    subdata1 = data['rides'][n]['line'] # This is the general information about the bus, such as the code, direction and name of the line
    subdata2 = data['rides'][n]['status']['deviation'] if data['rides'][n]['status']['deviation'] is not None else {'deviation_seconds': None, 'updated_at': None} # For some buses there is no information about the delay


    dict_data = {
        'id': data['rides'][n]['id'],
        'bus_code': subdata1['code'],
        'direction': subdata1['direction'],
        'name': subdata1['name'],
        'deviation_seconds': subdata2['deviation_seconds'],
        'updated_at': subdata2['updated_at'],
        'scheduled_timestamp': data['rides'][n]['status']['scheduled_timestamp']
        }
    
 
    return dict_data

In [ ]:
def extract_dep_delay_id_total(data): # Extract data for all the rides in the data

    list_of_dicts = []

    for i in range(len(data['rides'])):
        list_of_dicts.append(extract_dep_delay_id(data, i))

    return list_of_dicts

In [113]:
extract_dep_delay_id(data, 3)

{'id': 'ea707c63-d4b0-4e14-86a9-81469eddfb56',
 'bus_code': '482',
 'direction': 'Montpellier (Sabines)',
 'name': 'Venice - Montpellier',
 'deviation_seconds': 1220,
 'updated_at': '2026-05-13T10:53:34Z',
 'scheduled_timestamp': '2026-05-13T10:30:00Z'}

Now let us make one .csv file out of it. 
Btw, this is a nice pattern to learn. If the .csv file exists already (which in our case will be true), then it's good to check/know if the file is there, use it in an append mode, and also avoid writing the header more than 1 time. This we do in the following way:

In [122]:
file_exists = Path('/home/armankorajac/flixbus_tracker/data/processed/genoa_principe_departures.csv').exists() # Boolean to check if the file already exists, if not we will create it and write the header, otherwise we will append to it without writing the header

The arrays that we want to put:

In [119]:
processed_data= extract_dep_delay_id_total(data)
df = pd.DataFrame(processed_data)


0
length of rides:  18
1
length of rides:  18
2
length of rides:  18
3
length of rides:  18
4
length of rides:  18
5
length of rides:  18
6
length of rides:  18
7
length of rides:  18
8
length of rides:  18
9
length of rides:  18
10
length of rides:  18
11
length of rides:  18
12
length of rides:  18
13
length of rides:  18
14
length of rides:  18
15
length of rides:  18
16
length of rides:  18
17
length of rides:  18


In [120]:
df

,id,bus_code,direction,name,deviation_seconds,updated_at,scheduled_timestamp
0,1f831182-d3cd-49d6-aec5-ce6743908415,N718,Florence (Villa Costanza),Bordeaux - Nice - Florence,3617.0,2026-05-13T09:58:33Z,2026-05-13T08:55:00Z
1,56a3d5a0-5026-494a-a4e7-21e6742cd8a4,501,Naples (FS Park Stazione Centrale),Turin - Naples,232.0,2026-05-13T09:22:10Z,2026-05-13T09:15:00Z
2,8ab42e4d-ea1a-414c-9261-4bc0295004fb,798,Marseille (Saint-Charles),Marseille - Milan,3842.0,2026-05-13T10:47:10Z,2026-05-13T09:40:00Z
3,ea707c63-d4b0-4e14-86a9-81469eddfb56,482,Montpellier (Sabines),Venice - Montpellier,1220.0,2026-05-13T10:53:34Z,2026-05-13T10:30:00Z
4,30d07332-87b5-4723-829e-a1013810b55f,408,Bolzano South (Stazione FS),Genoa - Bozen,NaN,NaN,2026-05-13T11:25:00Z
5,40089f4b-d702-4a57-bfe8-fbeb9e73386a,N1138,Lisbon (Oriente),Lisbon - Milan,222.0,2026-05-13T11:08:44Z,2026-05-13T11:30:00Z
6,88e1da69-593e-4c80-bbd8-c57d8e882798,N456,Marseille (Saint-Antoine),Lublin - Milan - Marseille,467.0,2026-05-13T11:11:48Z,2026-05-13T12:30:00Z
7,a3a96a3b-a5fa-4feb-85e3-cd9d9808e6d3,494,Bolzano South (Stazione FS),Bolzano - Genoa,NaN,NaN,2026-05-13T12:50:00Z
8,ac572c34-3989-4c7c-92d1-84b8afdf1fd0,N719,Paris (Bercy Seine),Paris - Nice - Genova,NaN,2026-05-12T10:26:07Z,2026-05-13T13:05:00Z
9,459639cf-37e0-456a-98dd-5c61d0a21916,798,Milan (Lampugnano bus station),Marseille - Milan,1303.0,2026-05-13T11:13:06Z,2026-05-13T13:20:00Z


In [123]:
df.to_csv(
    "/home/armankorajac/flixbus_tracker/data/processed/genoa_principe_departures.csv",
    mode="a",
    header=not file_exists,
    index=False
)